# MIMIC Liver ACLF Prediction with Trajectory Features
Feature configurations: baseline, summary stats, trajectory probs, and combinations.

## Setup

In [13]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import os
import sys
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))
from notebook_utils import biomarker_summary_stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

✓ Imports successful


## Load Data

In [5]:
pred_with_probs_path = '../../../results/mimic/liver/liver_prediction_dataset_with_probs.csv'
outcome_path = '../../../results/mimic/liver/aclf_outcomes.csv'
ts_path = '../../../results/mimic/liver/bilirubin_timeseries.csv'

df = pd.read_csv(pred_with_probs_path)
outcomes = pd.read_csv(outcome_path)
bili_ts = pd.read_csv(ts_path)

print(f'✓ Loaded prediction dataset with probs: {len(df):,} rows')
print(f'✓ Outcomes: {len(outcomes):,} rows')
print(f'✓ Bilirubin TS: {len(bili_ts):,} rows')

df = df.merge(outcomes[['hadm_id', 'time_day', 'target_aclf']], on=['hadm_id', 'time_day'], how='left')
df = df[df['target_aclf'].notna()].copy()
df['target_aclf'] = df['target_aclf'].astype(int)

traj_cols = [c for c in df.columns if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]

print(f'Final dataset rows: {len(df):,}')
print(f'Outcome rate: {df["target_aclf"].mean():.1%}')
print(f'Trajectory columns: {len(traj_cols)}')

✓ Loaded prediction dataset with probs: 30,086 rows
✓ Outcomes: 15,405 rows
✓ Bilirubin TS: 32,339 rows
Final dataset rows: 15,405
Outcome rate: 4.2%
Trajectory columns: 3


## Feature Engineering

In [6]:
summary_5d = biomarker_summary_stats(bili_ts, 'bilirubin', lookback_days=5)
df = df.merge(summary_5d, on=['hadm_id', 'time_day'], how='left')

exclude_cols = {'hadm_id', 'time_day', 'charttime', 'admittime', 'dischtime', 'target_aclf'}
numeric_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
traj_cols = [c for c in numeric_cols if c.startswith('prob_') or c.endswith('_stable') or c.endswith('_gradual') or c.endswith('_rapid') or 'worsening' in c or 'improving' in c]
summary_cols = [c for c in numeric_cols if c.endswith('_5d') or '_trend_' in c or '_change_' in c]
base_cols = [c for c in numeric_cols if c not in traj_cols and c not in summary_cols]

static_name_tokens = ['age', 'gender', 'sex', 'baseline', 'admit', 'admission', 'ethnicity', 'race', 'height', 'weight', 'bmi']
static_cols = [c for c in base_cols if any(tok in c.lower() for tok in static_name_tokens)]
dynamic_cols = [c for c in base_cols if c not in static_cols]

feature_sets = {
    'Trajectory Only': traj_cols,
    'Summary Stats Only': summary_cols,
    'Trajectory + Summary Stats': traj_cols + summary_cols,
    'Static Only': static_cols,
    'Trajectory + Static': traj_cols + static_cols,
    'Summary Stats + Static': summary_cols + static_cols,
    'Trajectory + Summary Stats + Static': traj_cols + summary_cols + static_cols,
}

if len(dynamic_cols) > 0:
    feature_sets['Static + Dynamic'] = static_cols + dynamic_cols
    feature_sets['Trajectory + Static + Dynamic'] = traj_cols + static_cols + dynamic_cols
    feature_sets['Summary + Static + Dynamic'] = summary_cols + static_cols + dynamic_cols
    feature_sets['Trajectory + Summary + Static + Dynamic'] = traj_cols + summary_cols + static_cols + dynamic_cols

for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} features')

Trajectory Only: 3 features
Summary Stats Only: 6 features
Trajectory + Summary Stats: 9 features
Static Only: 2 features
Trajectory + Static: 5 features
Summary Stats + Static: 8 features
Trajectory + Summary Stats + Static: 11 features
Static + Dynamic: 86 features
Trajectory + Static + Dynamic: 89 features
Summary + Static + Dynamic: 92 features
Trajectory + Summary + Static + Dynamic: 95 features


## Model Comparison

In [9]:
df = df.drop_duplicates(subset=['hadm_id', 'time_day'])

In [10]:
df.shape

(15405, 104)

In [ ]:
models_to_evaluate = {
    'LogReg': lambda: LogisticRegression(max_iter=300, n_jobs=-1),
    'RF': lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=920),
    'HGB': lambda: HistGradientBoostingClassifier(random_state=920),
    'XGB': lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=-1, eval_metric='logloss'),
}

n_repeats = 10
n_splits = 5
results = []
metric_rows = []
curve_data = {}
y = df['target_aclf']
groups = df['hadm_id']

for model_name, model_fn in models_to_evaluate.items():
    dataset_model = df.copy()
    if len(traj_cols) > 0:
        dataset_model[traj_cols] = dataset_model.groupby('hadm_id')[traj_cols].ffill(limit=2)
    if model_name not in ['XGB', 'HGB']:
        if len(traj_cols) > 0:
            dataset_model[traj_cols] = dataset_model[traj_cols].fillna(0)
        if len(summary_cols) > 0:
            dataset_model[summary_cols] = dataset_model[summary_cols].fillna(0)

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            continue
        aucs, auprcs = [], []
        y_true_all, y_score_all = [], []
        for repeat in range(n_repeats):
            shuffle_idx = np.random.RandomState(seed=920 + repeat).permutation(len(dataset_model))
            dataset_repeat = dataset_model.iloc[shuffle_idx].reset_index(drop=True)
            y_repeat = y.iloc[shuffle_idx].reset_index(drop=True)
            groups_repeat = groups.iloc[shuffle_idx].reset_index(drop=True)
            gkf = GroupKFold(n_splits=n_splits)
            for train_idx, test_idx in gkf.split(dataset_repeat, y_repeat, groups_repeat):
                X_train = dataset_repeat.iloc[train_idx][feature_cols]
                X_test = dataset_repeat.iloc[test_idx][feature_cols]
                y_train = y_repeat.iloc[train_idx]
                y_test = y_repeat.iloc[test_idx]

                if model_name not in ['XGB', 'HGB']:
                    imputer = SimpleImputer(strategy='median')
                    X_train_imputed = imputer.fit_transform(X_train)
                    X_test_imputed = imputer.transform(X_test)
                else:
                    X_train_imputed = X_train
                    X_test_imputed = X_test

                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_imputed)
                X_test_scaled = scaler.transform(X_test_imputed)

                model = model_fn()
                model.fit(X_train_scaled, y_train)

                if hasattr(model, 'predict_proba'):
                    probs = model.predict_proba(X_test_scaled)[:, 1]
                else:
                    probs = model.decision_function(X_test_scaled)

                aucs.append(roc_auc_score(y_test, probs))
                auprcs.append(average_precision_score(y_test, probs))
                y_true_all.extend(y_test.tolist())
                y_score_all.extend(probs.tolist())

        curve_data[(model_name, feature_set_name)] = (np.array(y_true_all), np.array(y_score_all))
        for val in aucs:
            metric_rows.append({'model': model_name, 'feature_set': feature_set_name, 'metric': 'AUROC', 'value': val})
        for val in auprcs:
            metric_rows.append({'model': model_name, 'feature_set': feature_set_name, 'metric': 'AUPRC', 'value': val})

        print(f"{model_name} | {feature_set_name}: AUROC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}, AUPRC {np.mean(auprcs):.3f} ± {np.std(auprcs):.3f}")
        results.append({
            'model': model_name,
            'feature_set': feature_set_name,
            'auroc_mean': np.mean(aucs),
            'auroc_std': np.std(aucs),
            'auprc_mean': np.mean(auprcs),
            'auprc_std': np.std(auprcs),
        })

results_df = pd.DataFrame(results)
metrics_long = pd.DataFrame(metric_rows)
results_df

LogReg | Trajectory Only: AUROC 0.803 ± 0.012, AUPRC 0.117 ± 0.006
LogReg | Summary Stats Only: AUROC 0.887 ± 0.010, AUPRC 0.534 ± 0.029
LogReg | Trajectory + Summary Stats: AUROC 0.928 ± 0.011, AUPRC 0.581 ± 0.022
LogReg | Static Only: AUROC 0.736 ± 0.016, AUPRC 0.129 ± 0.024
LogReg | Trajectory + Static: AUROC 0.888 ± 0.011, AUPRC 0.419 ± 0.050
LogReg | Summary Stats + Static: AUROC 0.887 ± 0.004, AUPRC 0.535 ± 0.025
LogReg | Trajectory + Summary Stats + Static: AUROC 0.927 ± 0.010, AUPRC 0.583 ± 0.018
LogReg | Static + Dynamic: AUROC 0.953 ± 0.020, AUPRC 0.830 ± 0.033
LogReg | Trajectory + Static + Dynamic: AUROC 0.960 ± 0.015, AUPRC 0.848 ± 0.028
LogReg | Summary + Static + Dynamic: AUROC 0.968 ± 0.015, AUPRC 0.894 ± 0.034
LogReg | Trajectory + Summary + Static + Dynamic: AUROC 0.970 ± 0.015, AUPRC 0.896 ± 0.032
RF | Trajectory Only: AUROC 0.774 ± 0.018, AUPRC 0.106 ± 0.005
RF | Summary Stats Only: AUROC 0.928 ± 0.011, AUPRC 0.595 ± 0.025
RF | Trajectory + Summary Stats: AUROC 0.93

In [ ]:
summary_df = results_df.copy()
summary_df['AUROC'] = summary_df.apply(lambda r: f"{r.auroc_mean:.3f} ± {r.auroc_std:.3f}", axis=1)
summary_df['AUPRC'] = summary_df.apply(lambda r: f"{r.auprc_mean:.3f} ± {r.auprc_std:.3f}", axis=1)

for model in summary_df['model'].unique():
    print(f"\nModel: {model}")
    print(summary_df[summary_df['model'] == model][['feature_set', 'AUROC', 'AUPRC']].to_string(index=False))


Model: LogReg
                            feature_set         AUROC         AUPRC
                        Trajectory Only 0.803 ± 0.012 0.117 ± 0.006
                     Summary Stats Only 0.887 ± 0.010 0.534 ± 0.029
             Trajectory + Summary Stats 0.928 ± 0.011 0.581 ± 0.022
                            Static Only 0.736 ± 0.016 0.129 ± 0.024
                    Trajectory + Static 0.888 ± 0.011 0.419 ± 0.050
                 Summary Stats + Static 0.887 ± 0.004 0.535 ± 0.025
    Trajectory + Summary Stats + Static 0.927 ± 0.010 0.583 ± 0.018
                       Static + Dynamic 0.953 ± 0.020 0.830 ± 0.033
          Trajectory + Static + Dynamic 0.960 ± 0.015 0.848 ± 0.028
             Summary + Static + Dynamic 0.968 ± 0.015 0.894 ± 0.034
Trajectory + Summary + Static + Dynamic 0.970 ± 0.015 0.896 ± 0.032

Model: RF
                            feature_set         AUROC         AUPRC
                        Trajectory Only 0.774 ± 0.018 0.106 ± 0.005
                     S

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=metrics_long[metrics_long['metric'] == 'AUROC'], x='feature_set', y='value', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUROC Distribution by Feature Set and Model')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=metrics_long[metrics_long['metric'] == 'AUPRC'], x='feature_set', y='value', hue='model')
plt.xticks(rotation=45, ha='right')
plt.title('AUPRC Distribution by Feature Set and Model')
plt.tight_layout()
plt.show()

for model in results_df['model'].unique():
    plt.figure(figsize=(6, 5))
    for feature_set in results_df['feature_set'].unique():
        key = (model, feature_set)
        if key not in curve_data:
            continue
        y_true, y_score = curve_data[key]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        plt.plot(fpr, tpr, label=f"{feature_set} (AUC={auc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.title(f"ROC Curves - {model}")
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

for model in results_df['model'].unique():
    plt.figure(figsize=(6, 5))
    for feature_set in results_df['feature_set'].unique():
        key = (model, feature_set)
        if key not in curve_data:
            continue
        y_true, y_score = curve_data[key]
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        plt.plot(recall, precision, label=f"{feature_set} (AP={ap:.3f})")
    plt.title(f"PR Curves - {model}")
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

NameError: name 'metrics_long' is not defined

<Figure size 1200x500 with 0 Axes>

In [ ]:
sns.catplot(data=results_df, x='feature_set', y='auroc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC Liver: AUROC by Feature Set and Model')
plt.ylabel('AUROC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

sns.catplot(data=results_df, x='feature_set', y='auprc_mean', hue='model', kind='bar', height=4, aspect=2)
plt.title('MIMIC Liver: AUPRC by Feature Set and Model')
plt.ylabel('AUPRC')
plt.xlabel('Feature Set')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()